# Neural Network SMS Text Classifier

freeCodeCamp Machine Learning with Python project. Run all cells in order (Runtime → Run all).

In [ ]:
# import libraries
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget -q https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget -q https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

train_labels = (train_df['label'] == 'spam').astype('float32').values
test_labels = (test_df['label'] == 'spam').astype('float32').values

print(train_df['label'].value_counts())

In [ ]:
# Turn each message into TF-IDF scores over single words and word pairs
vectorizer = keras.layers.TextVectorization(max_tokens=20000, ngrams=2, output_mode='tf_idf')
vectorizer.adapt(train_df['message'].values)

train_x = vectorizer(train_df['message'].values)
test_x = vectorizer(test_df['message'].values)

model = keras.Sequential([
    keras.Input(shape=(train_x.shape[1],)),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Spam is only ~13% of messages, so weight it up to balance the classes
spam_weight = (train_labels == 0).sum() / (train_labels == 1).sum()

history = model.fit(
    train_x, train_labels,
    epochs=15,
    batch_size=32,
    validation_data=(test_x, test_labels),
    class_weight={0: 1.0, 1: spam_weight},
    verbose=2)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  probability = float(model.predict(vectorizer([pred_text]), verbose=0)[0][0])
  prediction = [probability, 'spam' if probability >= 0.5 else 'ham']

  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()